# Desafio 3: Modelo de lenguaje con tokenizacion por caracteres
## Alumno: Maxim Dorogov

### Consigna
- Seleccionar un corpus de texto sobre el cual entrenar el modelo de lenguaje.\
- Realizar el pre-procesamiento adecuado para tokenizar el corpus, estructurar el dataset y separar entre datos de entrenamiento y validación.
- Proponer arquitecturas de redes neuronales basadas en unidades recurrentes para implementar un modelo de lenguaje.
- Con el o los modelos que consideren adecuados, generar nuevas secuencias a partir de secuencias de contexto con las estrategias de greedy search y beam search determístico y estocástico. En este último caso observar el efecto de la temperatura en la generación de secuencias.

### Sugerencias
- Durante el entrenamiento, guiarse por el descenso de la perplejidad en los datos de validación para finalizar el entrenamiento. Para ello se provee un callback.
- Explorar utilizar SimpleRNN (celda de Elman), LSTM y GRU.
- rmsprop es el optimizador recomendado para la buena convergencia. No obstante se pueden explorar otros.

In [1]:
from tensorflow.keras.utils import pad_sequences
import numpy as np

2025-10-03 23:41:44.704324: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759545704.715819  113992 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759545704.719607  113992 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-03 23:41:44.732815: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Desarrollo

### Carga del dataset y tokenizacion

El dataset a utilizar es [Lovecraft Fiction](https://www.kaggle.com/datasets/bennijesus/lovecraft-fiction?resource=download), que contiene relatos, en idioma ingles, del autor H.P. Lovecraft.

In [2]:
# levantamos el archivo concat.txt el cual contiene todos los relatos concatenados.

DATA_PATH = '../desafio_2/data/concat.txt'
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    corpus = f.read()
# Elimino caracteres de salto de linea, retorno de carro y tabulacion
corpus = corpus.replace('\n', '')
corpus = corpus.replace('\r', '')
corpus = corpus.replace('\t', '')

print(f'Corpus length: {len(corpus)} characters')
print(f'Corpus sample: {corpus[:300]}')

Corpus length: 3851235 characters
Corpus sample: High up, crowning the grassy summit of a swelling mound whose sides are wooded near the basewith the gnarled trees of the primeval forest, stands the old chateau of my ancestors. For centuriesits lofty battlements have frowned down upon the wild and rugged countryside about, servingas a home and str


In [3]:
# Construimos vocabulario de caracteres y tokens

corpus = corpus.lower()
chars_vocab = set(corpus)

char2idx = {k: v for v,k in enumerate(chars_vocab)}
idx2char = {v: k for k,v in char2idx.items()}

# Tokenizamos

tokenized_text = [char2idx[ch] for ch in corpus]
print(tokenized_text[:20])
print([idx2char[c] for c in tokenized_text[:20]])
print(f'Vocabulary size: {len(chars_vocab)} characters')

[28, 17, 90, 28, 33, 3, 2, 34, 33, 41, 44, 53, 57, 50, 17, 50, 90, 33, 81, 28]
['h', 'i', 'g', 'h', ' ', 'u', 'p', ',', ' ', 'c', 'r', 'o', 'w', 'n', 'i', 'n', 'g', ' ', 't', 'h']
Vocabulary size: 97 characters


### Train / validation split: 80%/20%

In [4]:
CONTEXT_WINDOW_SIZE = 100
VAL_SPLIT = 0.2

val_sequence_qty = int(
    np.ceil(len(tokenized_text) * VAL_SPLIT / CONTEXT_WINDOW_SIZE))

# separamos la porción de texto utilizada en entrenamiento de la de validación.
train_text = tokenized_text[:-val_sequence_qty * CONTEXT_WINDOW_SIZE]
val_text = tokenized_text[-val_sequence_qty * CONTEXT_WINDOW_SIZE:]


print(f'Train size: {len(train_text)} tokens')
print(f'Validation size: {len(val_text)} tokens')

Train size: 3080935 tokens
Validation size: 770300 tokens


In [13]:
tokenized_sentences_val = [
    val_text[init*CONTEXT_WINDOW_SIZE:init*(CONTEXT_WINDOW_SIZE + 1)] 
    for init in range(val_sequence_qty)]

tokenized_sentences_train = [
    train_text[init:init + CONTEXT_WINDOW_SIZE] for
    init in range(len(train_text) - CONTEXT_WINDOW_SIZE + 1)]

X = np.array(list(tokenized_sentences_train)[:-1])
y = np.array(list(tokenized_sentences_train)[1:])